## Warehouse Location and Supply Chain

A company wants to open new warehouses to supply products to a set of 10 retail stores in a region. The goal is to minimize the total cost of opening warehouses and transporting goods to the stores.

- There are 20 potential sites for the warehouses.
- Each potential warehouse location has a fixed cost associated with opening it.
- Each retail store has a given demand that must be satisfied.
- The cost of transporting goods from each warehouse to each store is known.
- A warehouse, once opened, can supply goods to multiple stores, but each store must be supplied by exactly one warehouse.
- There is a constraint on the maximum number of warehouses that can be opened due to budget limitations.
- Each warehouse can send a maximum number of units.

Describe the parameters and variables of the problem and write the mathematical model.


Indexes:

i $\in$ I warehouse

j $\in$ J retail stores

l $\in$ location L

Parameters:

$c_{i,l}$ : cost of opening warehouse i in location l

$d_j$ : demand of retail store j

$c_{i,j}$ : transportation cost form i to j

$M_i$ : maximum number of warehouse that can be opened

$U_i$ : maximum number of units send by i

Variables:

$x_{i,l} = {0,1}$ 1 if warehouse i is opened in l

$z_{i,j}$ = amount of units from i to j

$y_{i,j}$ : if warehouse j is supplied by i

$$ min \ \ \sum_i \sum_j \sum_l c_{i,l} (x_{i,l} + c_{i,j})$$
$$ M_ i y_{i,j} \geq z_{i,j}$$
$$ sum_i y_{i,j} = 1 \ \ \forall j \in J $$
$$ sum_i sum_j x_{i,j} \leq M_i
$$ sum_j x_

In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

np.random.seed(42)
m = gp.Model("supplychain")
m.setParam("OutputFlag", 0)

num_warehouses = 20
num_stores = 10
I = list(range(num_warehouses))
J = list(range(num_stores))

fixed_costs = np.random.uniform(50000, 100000, num_warehouses)
demands = np.random.randint(100, 500, num_stores)
transport_costs = np.random.uniform(2, 15, (num_warehouses, num_stores))
max_warehouses = 5
max_units = 1500

x = m.addVars(I, vtype=GRB.BINARY, name="x")
z = m.addVars(I, J, vtype=GRB.CONTINUOUS, name="z")
y = m.addVars(I, J, vtype=GRB.BINARY, name="y")

for j in J:
    m.addConstr(gp.quicksum(y[i, j] for i in I) == 1)

m.addConstr(gp.quicksum(x[i] for i in I) <= max_warehouses)

for i in I:
    for j in J:
        m.addConstr(y[i, j] <= x[i])
        m.addConstr(z[i, j] == demands[j] * y[i, j])
    m.addConstr(gp.quicksum(z[i, j] for j in J) <= max_units * x[i])

m.setObjective(
    gp.quicksum(fixed_costs[i] * x[i] for i in I) + 
    gp.quicksum(transport_costs[i, j] * z[i, j] for i in I for j in J), 
    GRB.MINIMIZE
)
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"Total Minimized Cost: €{m.objVal:.2f}")
    print("Opened Warehouses:")
    for i in I:
        if x[i].x > 0.5:
            stores_served = [j for j in J if y[i,j].x > 0.5]
            print(f"- Warehouse {i} serving stores {stores_served} (Total units: {sum(z[i,j].x for j in stores_served):.0f})")


Total Minimized Cost: €121470.94
Opened Warehouses:
- Warehouse 6 serving stores [1, 2, 5, 6, 9] (Total units: 1323)
- Warehouse 10 serving stores [0, 3, 4, 7, 8] (Total units: 1434)
